In [ ]:
# --- Setup & imports ---
import os, sys, json, math, re, gc, random, h5py, warnings
from collections import defaultdict, Counter
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 120

warnings.filterwarnings("ignore", category=UserWarning)
torch.set_grad_enabled(False)

# --- Repro ---
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# --- Project root (edit if needed) ---
# PATH = "../../.."
PATH = "storage"

# --- Data loaders & datasets (your provided module) ---
from reetoolbox.dataloaders import (
    build_dataset, create_trainval_dict,
    NCTDataSet, PanNukeDataset, PandaDataset, PatchCamelyonDataset
)

# --- Small helpers ---
def ensure_dir(p):
    os.makedirs(p, exist_ok=True)
    return p

def to_pil(x):
    # x may be torch.Tensor CxHxW or PIL
    if isinstance(x, Image.Image): return x
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu()
        if x.ndim == 3:
            return transforms.ToPILImage()(x)
        elif x.ndim == 4 and x.size(0) == 1:
            return transforms.ToPILImage()(x[0])
    raise TypeError(f"Unsupported image type: {type(x)} | shape: {getattr(x,'shape',None)}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- Dataset configs (train/val) you gave ---
DATASET_CLASSES = {
    "NCT":            NCTDataSet,
    "PanNuke":        PanNukeDataset,
    "PANDA":          PandaDataset,
    "PatchCamelyon":  PatchCamelyonDataset,
}
DATASET_PARAMS = {
    "NCT": {
        "root_dir":            os.path.join(PATH, 'data', 'NCT', 'NCT-CRC-HE-100K'),
        "batch_size":          16,
        "trainval_multiplier": 0.1,
        "trainval_size":       0.3,
        "extra_kwargs":        dict(classification_mode='tum_vs_all'),
    },
    "PanNuke": {
        "root_dir":            os.path.join(PATH, 'data', 'PanNuke'),
        "batch_size":          16,
        "trainval_multiplier": 0.999,
        "trainval_size":       0.3,
        "extra_kwargs":        dict(folds=(1, 2), min_positive=5),
    },
    "PANDA": {
        "root_dir":            os.path.join(PATH, 'data', "PANDA"),
        "batch_size":          16,
        "trainval_multiplier": 0.154,
        "trainval_size":       0.3,
        "extra_kwargs":        dict(split='train'),
    },
    "PatchCamelyon": {
        "root_dir": os.path.join(PATH, 'data', 'PatchCamelyon'),
        "batch_size": 16,
        "trainval_multiplier": 0.0268,
        "trainval_size":       0.046,
        "extra_kwargs": {},
    },
}

# --- Test split params (from your snippet) ---
TEST_PARAMS = {
    "NCT": {
        "root_dir": os.path.join(PATH, 'data', 'NCT', 'CRC-VAL-HE-7K'),
        "batch_size": 16,
        "test_multiplier": 0.5,
        "transform": transforms.Compose([transforms.ToTensor()]),
        "dataset_kwargs": dict(classification_mode='tum_vs_all'),
    },
    "PanNuke": {
        "root_dir": os.path.join(PATH, 'data', 'PanNuke'),
        "batch_size": 16,
        "test_multiplier": 0.999,
        "transform": transforms.Compose([transforms.ToTensor()]),
        "dataset_kwargs": dict(folds=(3,), min_positive=5),
    },
    "PANDA": {
        "root_dir": os.path.join(PATH, 'data', 'PANDA'),
        "batch_size": 16,
        "test_multiplier": 0.6,  # spelled correctly
        "transform": transforms.Compose([transforms.ToTensor()]),
        "dataset_kwargs": dict(split='test'),
    },
    "PatchCamelyon": {
        "root_dir": os.path.join(PATH, 'data', 'PatchCamelyon'),
        "batch_size": 16,
        "test_multiplier": 0.135,
        "transform": transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()]),
        "dataset_kwargs": dict(split='test'),
    },
}

# --- Where to save stuff ---
def stat_dir(dataset_name, model_name):
    return ensure_dir(os.path.join(PATH, "Stats", dataset_name, model_name))

# --- Global knobs (feel free to tune) ---
CLUSTERS_PER_CLASS = 5     # k per class for KMeans
N_FOLDS = 5                # k-fold CV on clusters
PCA_DIM = 50               # pre-TSNE
TSNE_PERPLEXITY = 30
TSNE_ITER = 1000

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# HuggingFace login if needed
# ─────────────────────────────────────────────────────────────────────────────
from huggingface_hub import login
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Please set HF_TOKEN in your environment")
login(token=hf_token, add_to_git_credential=True)

# ─────────────────────────────────────────────────────────────────────────────
# Imports for models & transforms
# ─────────────────────────────────────────────────────────────────────────────
import timm
from timm.data import resolve_data_config, create_transform
from transformers import AutoImageProcessor, AutoModel, AutoModelForImageClassification

# EXAONEPath VT: make sure the GitHub path exists on disk
ex_path = os.path.join(PATH, "third_party", "EXAONEPath")
sys.path.append(ex_path)
from vision_transformer import VisionTransformer

# Wrap HF processors into a transform-like callable
class ProcessorTransform:
    def __init__(self, processor):
        self.processor = processor
    def __call__(self, img):
        # img is PIL
        return self.processor(img, return_tensors="pt")["pixel_values"].squeeze(0)

# Torch/timm transforms
uni_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.CenterCrop(size=(224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
giga_transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
exaone_transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
hoptimus_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.707223, 0.578729, 0.703617), std=(0.211883, 0.230117, 0.177517))
])
virchow_transform = transforms.Compose([
    transforms.Resize(size=224, interpolation=transforms.InterpolationMode.BICUBIC, max_size=None, antialias=True),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.4850, 0.4560, 0.4060), std=(0.2290, 0.2240, 0.2250))
])

# HF processors
resnet_processor   = AutoImageProcessor.from_pretrained("microsoft/resnet-18")
hibou_b_processor  = AutoImageProcessor.from_pretrained("histai/hibou-b", trust_remote_code=True)
hibou_L_processor  = AutoImageProcessor.from_pretrained("histai/hibou-L", trust_remote_code=True)
phikon_processor   = AutoImageProcessor.from_pretrained("owkin/phikon", trust_remote_code=True)
phikonv2_processor = AutoImageProcessor.from_pretrained("owkin/phikon-v2", trust_remote_code=True)

resnet_transform    = ProcessorTransform(resnet_processor)
hibou_b_transform   = ProcessorTransform(hibou_b_processor)
hibou_L_transform   = ProcessorTransform(hibou_L_processor)
phikon_transform    = ProcessorTransform(phikon_processor)
phikonv2_transform  = ProcessorTransform(phikonv2_processor)

def timm_transform(model):
    if hasattr(model, "pretrained_cfg"):
        timm_model = model
    elif hasattr(model, "backbone") and hasattr(model.backbone, "pretrained_cfg"):
        timm_model = model.backbone
    else:
        raise AttributeError("Cannot find .pretrained_cfg on the model or its backbone.")
    cfg = resolve_data_config(timm_model.pretrained_cfg, model=timm_model)
    return create_transform(**cfg)

def get_transform(model_name, model=None):
    if model_name in {"Virchow", "Virchow2", "H0-mini"} and model is not None:
        return timm_transform(model)
    # otherwise, use the predeclared mapping below
    return model_transforms[model_name]

model_transforms = {
    "ResNet18":     resnet_transform,  # or resnet50_transform for ResNet-50
    "ResNet34":     resnet_transform,  # or resnet50_transform for ResNet-50
    "ResNet50":     resnet_transform,  # or resnet50_transform for ResNet-50
    "ResNet101":     resnet_transform,  # or resnet50_transform for ResNet-50
    "ResNet152":     resnet_transform,  # or resnet50_transform for ResNet-50
    "UNI":        uni_transform,
    "UNI2":        uni_transform,
    "GigaPath":   giga_transform,
    "Virchow":    virchow_transform,   # special: see below
    "Virchow2":   virchow_transform,   # special: see below
    "EXAONEPath": exaone_transform,
    "Hibou-B":    hibou_b_transform,
    "Hibou-L":    hibou_L_transform,
    "H-Optimus-0": hoptimus_transform,
    "H-Optimus-1": hoptimus_transform,
    "H0-mini":    hoptimus_transform,
    "Phikon v1":  phikon_transform,
    "Phikon v2":  phikonv2_transform,
}

# --- Your model configs exactly as provided ---
model_configs = [
    {
        "name": "ResNet18",
        "enabled": True,
        "init": lambda: AutoModelForImageClassification.from_pretrained("microsoft/resnet-18").eval().to(DEVICE),
        "transform": resnet_transform,
        "embedding_fn": lambda model, tensor: model(pixel_values=tensor.to(DEVICE)).logits.detach().cpu().numpy(),
    },
    {
        "name": "ResNet34",
        "enabled": True,
        "init": lambda: AutoModelForImageClassification.from_pretrained("microsoft/resnet-34").eval().to(DEVICE),
        "transform": resnet_transform,
        "embedding_fn": lambda model, tensor: model(pixel_values=tensor.to(DEVICE)).logits.detach().cpu().numpy(),
    },
    {
        "name": "ResNet50",
        "enabled": True,
        "init": lambda: timm.create_model("resnet50", pretrained=True).eval().to(DEVICE),
        "transform": resnet_transform,
        "embedding_fn": lambda model, tensor: model(tensor.to(DEVICE)).detach().cpu().numpy(),
    },
    {
        "name": "ResNet101",
        "enabled": True,
        "init": lambda: timm.create_model("resnet101", pretrained=True).eval().to(DEVICE),
        "transform": resnet_transform,
        "embedding_fn": lambda model, tensor: model(tensor.to(DEVICE)).detach().cpu().numpy(),
    },
    {
        "name": "ResNet152",
        "enabled": True,
        "init": lambda: timm.create_model("resnet152", pretrained=True).eval().to(DEVICE),
        "transform": resnet_transform,
        "embedding_fn": lambda model, tensor: model(tensor.to(DEVICE)).detach().cpu().numpy(),
    },
    {
        "name": "UNI",
        "enabled": True,
        "init": lambda: timm.create_model("hf-hub:MahmoodLab/uni", pretrained=True, init_values=1e-5, dynamic_img_size=True).eval().to(DEVICE),
        "transform": uni_transform,
        "embedding_fn": lambda model, tensor: model(tensor.to(DEVICE)).detach().cpu().numpy(),
    },
    {
        "name": "UNI2",
        "enabled": True,
        "init": lambda: timm.create_model(
            "hf-hub:MahmoodLab/UNI2-h",
            pretrained=True,
            img_size=224, patch_size=14, depth=24, num_heads=24,
            init_values=1e-5, embed_dim=1536, mlp_ratio=2.66667*2, num_classes=0, no_embed_class=True,
            mlp_layer=timm.layers.SwiGLUPacked, act_layer=torch.nn.SiLU, reg_tokens=8, dynamic_img_size=True,
        ).eval().to(DEVICE),
        "transform": uni_transform,
        "embedding_fn": lambda model, tensor: model(tensor.to(DEVICE)).detach().cpu().numpy(),
    },
    {
        "name": "GigaPath",
        "enabled": True,
        "init": lambda: timm.create_model("hf_hub:prov-gigapath/prov-gigapath", pretrained=True).eval().to(DEVICE),
        "transform": giga_transform,
        "embedding_fn": lambda model, tensor: model(tensor.to(DEVICE)).detach().cpu().numpy(),
    },
    {
        "name": "Virchow",
        "enabled": True,
        "init": lambda: timm.create_model(
            "hf-hub:paige-ai/Virchow", pretrained=True,
            mlp_layer=timm.layers.SwiGLUPacked, act_layer=torch.nn.SiLU,
        ).eval().to(DEVICE),
        "transform": virchow_transform,
        "embedding_fn": lambda model, tensor: torch.cat([
            model(tensor.to(DEVICE))[:, 0],
            model(tensor.to(DEVICE))[:, 1:].mean(1)
        ], dim=-1).detach().cpu().numpy(),
    },
    {
        "name": "Virchow2",
        "enabled": True,
        "init": lambda: timm.create_model(
            "hf-hub:paige-ai/Virchow2", pretrained=True,
            mlp_layer=timm.layers.SwiGLUPacked, act_layer=torch.nn.SiLU,
        ).eval().to(DEVICE),
        "transform": virchow_transform,
        "embedding_fn": lambda model, tensor: torch.cat([
            model(tensor.to(DEVICE))[:, 0],
            model(tensor.to(DEVICE))[:, 5:].mean(1)
        ], dim=-1).detach().cpu().numpy(),
    },
    {
        "name": "EXAONEPath",
        "enabled": True,
        "init": lambda: VisionTransformer.from_pretrained("LGAI-EXAONE/EXAONEPath", use_auth_token=hf_token).eval().to(DEVICE),
        "transform": exaone_transform,
        "embedding_fn": lambda model, tensor: model(tensor.to(DEVICE)).squeeze().detach().cpu().numpy(),
    },
    {
        "name": "Hibou-B",
        "enabled": True,
        "init": lambda: AutoModel.from_pretrained("histai/hibou-b", trust_remote_code=True).eval().to(DEVICE),
        "transform": hibou_b_transform,
        "embedding_fn": lambda model, tensor: model(pixel_values=tensor.to(DEVICE)).last_hidden_state[:, 0, :].detach().cpu().numpy(),
    },
    {
        "name": "Hibou-L",
        "enabled": True,
        "init": lambda: AutoModel.from_pretrained("histai/hibou-L", trust_remote_code=True).eval().to(DEVICE),
        "transform": hibou_L_transform,
        "embedding_fn": lambda model, tensor: model(pixel_values=tensor.to(DEVICE)).last_hidden_state[:, 0, :].detach().cpu().numpy(),
    },
    {
        "name": "Phikon v1",
        "enabled": True,
        "init": lambda: AutoModel.from_pretrained("owkin/phikon", trust_remote_code=True).eval().to(DEVICE),
        "transform": phikon_transform,
        "embedding_fn": lambda model, tensor: model(pixel_values=tensor.to(DEVICE)).last_hidden_state[:, 0, :].detach().cpu().numpy(),
    },
    {
        "name": "Phikon v2",
        "enabled": True,
        "init": lambda: AutoModel.from_pretrained("owkin/phikon-v2", trust_remote_code=True).eval().to(DEVICE),
        "transform": phikonv2_transform,
        "embedding_fn": lambda model, tensor: model(pixel_values=tensor.to(DEVICE)).last_hidden_state[:, 0, :].detach().cpu().numpy(),
    },
    {
        "name": "H-Optimus-0",
        "enabled": True,
        "init": lambda: timm.create_model("hf-hub:bioptimus/H-optimus-0", pretrained=True, init_values=1e-5, dynamic_img_size=False).eval().to(DEVICE),
        "transform": hoptimus_transform,
        "embedding_fn": lambda model, tensor: model(tensor.to(DEVICE)).detach().cpu().numpy(),
    },
    {
        "name": "H-Optimus-1",
        "enabled": True,
        "init": lambda: timm.create_model("hf-hub:bioptimus/H-optimus-1", pretrained=True, init_values=1e-5, dynamic_img_size=False).eval().to(DEVICE),
        "transform": hoptimus_transform,
        "embedding_fn": lambda model, tensor: model(tensor.to(DEVICE)).detach().cpu().numpy(),
    },
    {
        "name": "H0-mini",
        "enabled": True,
        "init": lambda: timm.create_model(
            "hf-hub:bioptimus/H0-mini", pretrained=True,
            mlp_layer=timm.layers.SwiGLUPacked, act_layer=torch.nn.SiLU
        ).eval().to(DEVICE),
        "transform": hoptimus_transform,
        "embedding_fn": lambda model, tensor: torch.cat([
            model(tensor.to(DEVICE))[:, 0],          # CLS token
            model(tensor.to(DEVICE))[:, 1:].mean(1)  # mean of patch tokens
        ], dim=-1).detach().cpu().numpy(),
    }
]

In [ ]:
def build_loaders_for_dataset(dataset_name: str):
    # train/val via provided helper
    params = DATASET_PARAMS[dataset_name]
    dataset_class = DATASET_CLASSES[dataset_name]

    # Keep transforms as ToTensor() so DataLoader can batch; we will re-transform per-model when needed.
    tv_transform = transforms.Compose([transforms.ToTensor()])

    dl_trainval = create_trainval_dict(
        dataset_class=dataset_class,
        root_dir=params["root_dir"],
        batch_size=params["batch_size"],
        transform=tv_transform,
        trainval_multiplier=params["trainval_multiplier"],
        trainval_size=params["trainval_size"],
        shuffle=False,
        **params["extra_kwargs"]
    )

    # test via your sampled helper
    tconf = TEST_PARAMS[dataset_name]
    test_data, test_loader = build_dataset(
        dataset_class=dataset_class,
        root_dir=tconf["root_dir"],
        batch_size=tconf["batch_size"],
        test_multiplier=tconf["test_multiplier"],
        transform=tconf["transform"],
        shuffle=False,
        **tconf["dataset_kwargs"]
    )

    loaders = {'train': dl_trainval['train'], 'val': dl_trainval['val'], 'test': test_loader}
    return loaders

In [ ]:
def prepare_batch_for_model(img_batch, transform_callable):
    # img_batch: Tensor [B,C,H,W] or list of Tensors
    if isinstance(img_batch, torch.Tensor):
        imgs = [img_batch[i] for i in range(img_batch.size(0))]
    else:
        imgs = list(img_batch)

    processed = []
    for t in imgs:
        pil = to_pil(t)       # ensure PIL for processor-based transforms
        out = transform_callable(pil)
        if isinstance(out, torch.Tensor):
            processed.append(out)
        else:
            # HF ProcessorTransform returns torch.Tensor already; if numpy, convert
            processed.append(torch.from_numpy(out) if isinstance(out, np.ndarray) else out)

    batch = torch.stack(processed, dim=0)
    return batch

def batched_iter(loader):
    for imgs, labels, names in loader:
        # labels may be list or tensor
        if isinstance(labels, torch.Tensor):
            labels = labels.detach().cpu().numpy().tolist()
        else:
            labels = list(labels)
        # names is list
        yield imgs, labels, list(names)

def extract_embeddings(model_cfg, loaders_by_split, dataset_name: str):
    model_name = model_cfg["name"]
    save_root  = os.path.join(stat_dir(dataset_name, model_name), "distribution_shift")
    ensure_dir(os.path.join(save_root, "embeddings"))

    # init model
    model = model_cfg["init"]()
    # select correct transform (some depend on model instance)
    model_transform = get_transform(model_name, model=model)

    all_feats, all_labels, all_names, all_splits = [], [], [], []

    for split, loader in loaders_by_split.items():
        for imgs, labels, names in tqdm(batched_iter(loader), desc=f"{dataset_name}/{model_name} [{split}]"):
            # per-model transform -> batch
            batch = prepare_batch_for_model(imgs, model_transform)
            # embedding fn expects a batch tensor in the correct format
            feats = model_cfg["embedding_fn"](model, batch)
            feats = np.asarray(feats)
            if feats.ndim == 1:
                feats = feats[None, :]
            all_feats.append(feats)
            all_labels.extend(labels)
            all_names.extend(names)
            all_splits.extend([split]*len(labels))

    # stack & save
    feats = np.concatenate(all_feats, axis=0) if len(all_feats) else np.zeros((0, 1))
    meta = pd.DataFrame({
        "name": all_names,
        "label": all_labels,
        "split": all_splits,
        "dataset": dataset_name,
        "model": model_name,
    })

    # Save HDF5 + CSV
    h5_path = os.path.join(save_root, "embeddings", "embeddings.h5")
    with h5py.File(h5_path, "w") as h5:
        h5.create_dataset("features", data=feats, compression="gzip")
        h5.create_dataset("labels", data=np.array(all_labels, dtype=np.int64))
        # store names as fixed-length ASCII
        str_dt = h5py.string_dtype(encoding='utf-8')
        h5.create_dataset("names", data=np.array(all_names, dtype=object), dtype=str_dt)
        h5.create_dataset("splits", data=np.array(all_splits, dtype=object), dtype=str_dt)

    meta_path = os.path.join(save_root, "embeddings", "metadata.csv")
    meta.to_csv(meta_path, index=False)

    # free GPU
    del model; torch.cuda.empty_cache(); gc.collect()
    return feats, meta

In [ ]:
def cluster_by_class(feats: np.ndarray, labels: List[int], k_per_class=CLUSTERS_PER_CLASS):
    labels = np.array(labels)
    uniq = np.unique(labels)
    N = feats.shape[0]
    cluster_ids = np.full(N, fill_value=-1, dtype=int)
    global_id = 0
    per_cluster_sizes = {}

    for cls in uniq:
        idx = np.where(labels == cls)[0]
        if len(idx) == 0:
            continue
        k = min(k_per_class, len(idx))  # avoid k > n
        if k <= 1:
            cluster_ids[idx] = global_id
            per_cluster_sizes[global_id] = len(idx)
            global_id += 1
            continue

        km = KMeans(n_clusters=k, n_init=10, random_state=SEED)
        cid = km.fit_predict(feats[idx])
        for local in range(k):
            sel = idx[cid == local]
            cluster_ids[sel] = global_id
            per_cluster_sizes[global_id] = len(sel)
            global_id += 1

    return cluster_ids, per_cluster_sizes

def save_clusters(meta: pd.DataFrame, cluster_ids: np.ndarray, dataset_name: str, model_name: str):
    out_dir = os.path.join(stat_dir(dataset_name, model_name), "distribution_shift")
    df = meta.copy()
    df["cluster_id"] = cluster_ids
    df.to_csv(os.path.join(out_dir, "clusters.csv"), index=False)
    # also json summary of cluster sizes
    sizes = df["cluster_id"].value_counts().sort_index().to_dict()
    with open(os.path.join(out_dir, "cluster_sizes.json"), "w") as f:
        json.dump({str(k): int(v) for k,v in sizes.items()}, f, indent=2)

def assign_folds_from_clusters(meta_with_clusters: pd.DataFrame, n_folds=N_FOLDS):
    """
    Greedy assignment of whole clusters to folds to balance total size and preserve class balance.
    Operates per-class to keep folds balanced across classes.
    """
    df = meta_with_clusters.copy()
    folds = np.full(len(df), -1, dtype=int)

    for cls in sorted(df["label"].unique()):
        sub = df[df["label"] == cls]
        # cluster -> list(idx)
        groups = sub.groupby("cluster_id").indices
        # sort clusters by decreasing size
        clusters_sorted = sorted(groups.items(), key=lambda kv: -len(kv[1]))
        fold_sizes = [0]*n_folds

        for cid, idxs in clusters_sorted:
            # pick fold with smallest current size for this class
            f = int(np.argmin(fold_sizes))
            fold_sizes[f] += len(idxs)
            # map global indices
            global_idxs = sub.index.values[idxs]
            folds[global_idxs] = f

    # sanity: all assigned?
    assert np.all(folds >= 0)
    df["fold"] = folds
    return df

def save_folds(df_folds: pd.DataFrame, dataset_name: str, model_name: str):
    out_dir = os.path.join(stat_dir(dataset_name, model_name), "distribution_shift")
    ensure_dir(os.path.join(out_dir, "folds"))
    df_folds.to_csv(os.path.join(out_dir, "folds", f"fold_assignments_k{N_FOLDS}.csv"), index=False)
    # fold stats
    stats = df_folds.groupby(["fold","label"]).size().unstack(fill_value=0)
    stats.to_csv(os.path.join(out_dir, "folds", f"fold_class_counts_k{N_FOLDS}.csv"))

In [ ]:
# REPLACE any previous run_tsne_and_plot* with this
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

def run_tsne_and_plot_folds(
    feats,
    meta_with_folds,          # must contain columns: ['label','fold', ...]
    dataset_name,
    model_name,
    perplexity=30,
    n_iter=1000,
    show_counts=False         # set True if you want "(n=…)" in legend labels
):
    """
    Visualize t-SNE colored by FOLD (not cluster). Also outputs a class-colored t-SNE.
    - NO PCA pre-reduction.
    - Normal matplotlib legend.
    - Axes and tick values are shown.
    """
    out_dir = ensure_dir(os.path.join(stat_dir(dataset_name, model_name), "distribution_shift", "tsne"))

    # ---- t-SNE directly on feats (NO PCA) ----
    ts = TSNE(
        n_components=2,
        perplexity=perplexity,
        max_iter=n_iter,
        random_state=SEED,
        init="random",            # avoid PCA init to respect "no PCA"
        learning_rate="auto"
    )
    X2 = ts.fit_transform(feats)

    df = meta_with_folds.copy()
    df["x"], df["y"] = X2[:, 0], X2[:, 1]

    # ---------- Plot 1: by FOLD ----------
    plt.figure(figsize=(8, 6))
    folds = sorted(df["fold"].unique())
    cmap = plt.get_cmap("tab20")
    for i, f in enumerate(folds):
        sub = df[df["fold"] == f]
        label = f"Fold {f}" + (f" (n={len(sub)})" if show_counts else "")
        plt.scatter(sub["x"], sub["y"], s=8, alpha=0.85, color=cmap(i % 20), label=label)
    plt.title(f"{model_name} {dataset_name} t-SNE by Fold")
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    plt.legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "tsne_byfold.png"), dpi=150)
    plt.close()

    # ---------- Plot 2: by CLASS (blue/red) ----------
    plt.figure(figsize=(8, 6))
    class_color = {0: "tab:blue", 1: "tab:red"}
    class_name  = {0: "FALSE/Normal", 1: "TRUE/Tumor"}
    for c in sorted(df["label"].unique()):
        sub = df[df["label"] == c]
        label = class_name.get(c, f"Class {c}") + (f" (n={len(sub)})" if show_counts else "")
        plt.scatter(sub["x"], sub["y"], s=8, alpha=0.85, color=class_color.get(c, "k"), label=label)
    plt.title(f"{model_name} {dataset_name} t-SNE by Class")
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    plt.legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "tsne_byclass.png"), dpi=150)
    plt.close()


In [ ]:
DATASETS = ["NCT", "PanNuke", "PANDA", "PatchCamelyon"]  # edit to subset

for dataset_name in DATASETS:
    loaders = build_loaders_for_dataset(dataset_name)

    for cfg in model_configs:
        if not cfg.get("enabled", True):
            continue

        model_name = cfg["name"]
        print(f"\n=== {dataset_name} :: {model_name} ===")
        # 1) Embeddings
        feats, meta = extract_embeddings(cfg, loaders, dataset_name)
        # 2) Clustering (per class)
        cluster_ids, size_map = cluster_by_class(feats, meta["label"].tolist(), k_per_class=CLUSTERS_PER_CLASS)
        save_clusters(meta, cluster_ids, dataset_name, model_name)
        meta_c = meta.copy(); meta_c["cluster_id"] = cluster_ids
        # 3) Fold assignment from clusters
        meta_folds = assign_folds_from_clusters(meta_c, n_folds=N_FOLDS)
        save_folds(meta_folds, dataset_name, model_name)
        # 4) t-SNE plots
        run_tsne_and_plot_folds(
                                feats,
                                meta_folds,                 # has the 'fold' column
                                dataset_name,
                                model_name,
                                perplexity=TSNE_PERPLEXITY,
                                n_iter=TSNE_ITER,
                                show_counts=True           # flip to True if you want (n=...) shown
                            )


print("Done.")